In [9]:
import pandas as pd
import numpy as np
import re
from astropy.coordinates import SkyCoord
import astropy.units as u


def read_sextractor_ascii_head(cat_file):
    colnames = []

    with open(cat_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                m = re.match(r"#\s*\d+\s+(\S+)", line)
                if m:
                    colnames.append(m.group(1))
            else:
                break

    df = pd.read_csv(
        cat_file,
        comment="#",
        delim_whitespace=True,
        names=colnames
    )

    return df


# ============================================================
# USER INPUT
# ============================================================
CAT_FILE = "/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y_masked.cat"

target = SkyCoord(
    ra="23:48:33.33",
    dec="-30:54:10.23",
    unit=(u.hourangle, u.deg),
    frame="fk5"
)

MATCH_RADIUS = 1.0 * u.arcsec   # change if you want


# ============================================================
# LOAD CATALOG
# ============================================================
df = read_sextractor_ascii_head(CAT_FILE)

coords = SkyCoord(
    ra=df["ALPHA_J2000"].values * u.deg,
    dec=df["DELTA_J2000"].values * u.deg,
    frame="fk5"
)

# ============================================================
# MATCH
# ============================================================
sep = target.separation(coords)

mask = sep < MATCH_RADIUS

if np.any(mask):
    print("SOURCE FOUND")
    print(df.loc[mask])
    print("\nAngular separation (arcsec):")
    print(sep[mask].arcsec)
else:
    print("SOURCE NOT FOUND within", MATCH_RADIUS)


/var/folders/9v/db3j63px68q3f33kdb7_7wbc0000gn/T/ipykernel_19665/624481851.py:20: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


SOURCE FOUND
          NUMBER     X_IMAGE     Y_IMAGE  ALPHA_J2000  DELTA_J2000  MAG_APER  \
2044678  2260218  13332.4297  12252.8643   357.138839   -30.902851   -7.1968   

         MAGERR_APER  MAG_AUTO  MAGERR_AUTO  FLAGS  
2044678       0.0223   -7.4036       0.0215      0  

Angular separation (arcsec):
[0.11627186]


In [3]:
df = pd.read_csv("/Users/aishwarya/Desktop/Lyman_alpha_2/CAT/LBG_Y_Y.cat")
print(df.columns)


Index(['#   1 NUMBER                 Running object number                                     '], dtype='object')


In [10]:
from astropy.io import fits
from astropy.wcs import WCS

files = [
    "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Y.fits",
    "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Z.fits",
    "/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_I.fits"
]

for f in files:
    hdr = fits.getheader(f)
    wcs = WCS(hdr)
    print(f"\nFile: {f}")
    print("CRVAL (RA, DEC):", wcs.wcs.crval)
    print("CRPIX (X, Y):", wcs.wcs.crpix)



File: /Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Y.fits
CRVAL (RA, DEC): [357.188  -31.0392]
CRPIX (X, Y): [12770. 10435.]

File: /Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Z.fits
CRVAL (RA, DEC): [357.188  -31.0392]
CRPIX (X, Y): [12770. 10435.]

File: /Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_I.fits
CRVAL (RA, DEC): [357.188  -31.0392]
CRPIX (X, Y): [12770. 10435.]


In [11]:
from astropy.coordinates import SkyCoord
import astropy.units as u

ra_test = 357.8
dec_test = -30.8

coord = SkyCoord(ra_test*u.deg, dec_test*u.deg)

for f in files:
    hdr = fits.getheader(f)
    wcs = WCS(hdr)
    x, y = wcs.world_to_pixel(coord)
    print(f"{f}: x={x:.3f}, y={y:.3f}")


/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Y.fits: x=5759.665, y=13604.183
/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_Z.fits: x=5759.665, y=13604.183
/Users/aishwarya/Desktop/Lyman_alpha_2/Background_check/BG_Y_I.fits: x=5759.665, y=13604.183
